In [2]:
import pandas as pd
import re

In [3]:
# Load all_articles.csv into a DataFrame
df = pd.read_csv('data/00_raw/all_articles.csv')
df

,title,summary,section,keywords,published_date,url
0,"Paid Notice: Deaths BAUER, LILLIAN","BAUER-Lillian. December 30, 1999. Beloved wife...",Archives,"['Bauer, Lillian']",2000-01-01 05:00:00+00:00,https://www.nytimes.com/2000/01/01/classified/...
1,"Paid Notice: Deaths GLUCK, SAMUEL E.","GLUCK-Samuel E.. At age 74. On December 31, 19...",Archives,"['GLUCK, SAMUEL E.']",2000-01-01 05:00:00+00:00,https://www.nytimes.com/2000/01/01/classified/...
2,Folds of Newspapers Yield Symbols of Peace,"Joshua Davis, high school student from Montcla...",New York,"['Montclair (NJ)', 'DAVIS, JOSHUA', 'Newspaper...",2000-01-01 05:00:00+00:00,https://www.nytimes.com/2000/01/01/nyregion/fo...
3,"T. F. Lambert Jr., 85, Lawyer at Nuremberg","Thomas Francis Lambert Jr, law professor and d...",U.S.,"['Lambert, Thomas Francis Jr', 'Biographical I...",2000-01-01 05:00:00+00:00,https://www.nytimes.com/2000/01/01/us/t-f-lamb...
4,"Paid Notice: Deaths SMITH, KATHERINE ANNE","SMITH-Katherine Anne. Died on December 30, 199...",Archives,"['Smith, Katherine Anne']",2000-01-01 05:00:00+00:00,https://www.nytimes.com/2000/01/01/classified/...
...,...,...,...,...,...,...
299604,Environmental groups demand EPA to start monit...,A new legal petition filed by more than 170 to...,Environment,"['US Environmental Protection Agency', 'Plasti...",2024-12-01 13:00:07+00:00,https://www.theguardian.com/environment/2024/d...
299605,"Land degradation expanding by 1m sq km a year,...",Land degradation is expanding worldwide at the...,Environment,"['Desertification', 'Farming', 'Deforestation'...",2024-12-01 12:00:07+00:00,https://www.theguardian.com/environment/2024/d...
299606,"‘If I’m sent to Japan, I’m not coming home’: j...",The humpback whales watched by Paul Watson fro...,Environment,"['Sea Shepherd Conservation Society', 'Whales'...",2024-12-01 08:00:01+00:00,https://www.theguardian.com/environment/2024/d...
299607,Cheaper loans on table to urge UK motorists to...,There is “no route to net zero” that ignores t...,Environment,"['Electric, hybrid and low-emission cars', 'Au...",2024-12-01 07:00:50+00:00,https://www.theguardian.com/environment/2024/d...


## Filter sections

In [4]:
# Check all sections, remove nan, sort them, and print them out
sections = df['section'].dropna().unique()
sections.sort()
sections

array(['Admin', 'Archives', 'Arts', 'At Home', 'Automobiles', 'Blogs',
       'Book Review', 'Books', 'Booming', 'Briefing', 'Burst',
       'Business Day', 'Climate', 'Corrections', 'Crosswords & Games',
       'Education', 'Environment', 'Fashion & Style', 'Food',
       'Great Homes & Destinations', 'Headway', 'Health', 'Home & Garden',
       'International Home', 'Job Market', 'Lens', 'Magazine', 'Movies',
       'Multimedia/Photos', 'New York', 'Obituaries', 'Opinion',
       'Parenting', 'Real Estate', 'Science', 'Smarter Living',
       'Special Series', 'Sports', 'Style', 'T Magazine', 'Technology',
       'The New York Times Presents', 'The Upshot', 'Theater',
       'Times Insider', 'Today’s Paper', 'Travel', 'U.S.', 'Universal',
       'Washington', 'Watching', 'Well', 'World', 'Your Money'],
      dtype=object)

In [5]:
# Analyze section distribution
section_counts = df['section'].value_counts(dropna=False)
print("=== Section Distribution ===")
print(section_counts)
print(f"\nTotal sections: {len(sections)}")
print(f"Articles with no section: {df['section'].isna().sum()}")

=== Section Distribution ===
section
New York                       69610
Business Day                   56914
Environment                    55215
Sports                         31692
Archives                       24922
Books                          20381
Technology                     11990
Arts                            8410
U.S.                            4376
World                           2852
Your Money                      2775
Obituaries                      1895
Real Estate                     1422
Job Market                      1098
The Upshot                       838
Theater                          822
Health                           816
Movies                           677
Automobiles                      371
Science                          363
Education                        353
Travel                           298
Style                            292
Fashion & Style                  279
Climate                          253
Smarter Living                   203
C

In [6]:
# Define sections to INCLUDE (business-relevant for company mentions)
INCLUDE_SECTIONS = [
    # Core Business
    'Business Day',
    'Your Money',
    'Job Market',
    
    # Relevant companies possible, have to use entity linking to filter out non-company mentions
    'Technology',
    'Automobiles',
    'Environment', 
    'Climate',
    'Health', 
    'Real Estate',
    'Science',  
    'Smarter Living', # discusses consumer products and lifestyle, which may include company mentions

    
    # News sections 
    'U.S.',
    'World',
    'New York',
    'Washington',
    'The Upshot'
    
    
]

# Analyze which sections we're including
print("=== SECTIONS TO INCLUDE ===")
section_counts = df['section'].value_counts(dropna=False)

for section in INCLUDE_SECTIONS:
    if section in section_counts.index:
        count = section_counts[section]
        pct = count / len(df) * 100
        print(f"- {section}: {count} articles ({pct:.2f}%)")
    else:
        print(f"- {section}: NOT FOUND in data")

total_to_keep = df[df['section'].isin(INCLUDE_SECTIONS)].shape[0]
print(f"\nTotal articles to include: {total_to_keep} ({total_to_keep/len(df)*100:.2f}%)")

=== SECTIONS TO INCLUDE ===
- Business Day: 56914 articles (19.00%)
- Your Money: 2775 articles (0.93%)
- Job Market: 1098 articles (0.37%)
- Technology: 11990 articles (4.00%)
- Automobiles: 371 articles (0.12%)
- Environment: 55215 articles (18.43%)
- Climate: 253 articles (0.08%)
- Health: 816 articles (0.27%)
- Real Estate: 1422 articles (0.47%)
- Science: 363 articles (0.12%)
- Smarter Living: 203 articles (0.07%)
- U.S.: 4376 articles (1.46%)
- World: 2852 articles (0.95%)
- New York: 69610 articles (23.23%)
- Washington: 40 articles (0.01%)
- The Upshot: 838 articles (0.28%)

Total articles to include: 209136 (69.80%)


In [7]:
# Sections we're EXCLUDING 
exclude_sections = [s for s in section_counts.index if s not in INCLUDE_SECTIONS and pd.notna(s)]

print("\n=== SECTIONS TO EXCLUDE (not in include list) ===")
for section in sorted(exclude_sections):
    count = section_counts[section]
    pct = count / len(df) * 100
    print(f"- {section}: {count} articles ({pct:.2f}%)")

total_to_exclude = df[df['section'].isin(exclude_sections)].shape[0]
print(f"\nTotal articles to exclude: {total_to_exclude} ({total_to_exclude/len(df)*100:.2f}%)")


=== SECTIONS TO EXCLUDE (not in include list) ===
- Admin: 1 articles (0.00%)
- Archives: 24922 articles (8.32%)
- Arts: 8410 articles (2.81%)
- At Home: 1 articles (0.00%)
- Blogs: 12 articles (0.00%)
- Book Review: 2 articles (0.00%)
- Books: 20381 articles (6.80%)
- Booming: 6 articles (0.00%)
- Briefing: 6 articles (0.00%)
- Burst: 2 articles (0.00%)
- Corrections: 11 articles (0.00%)
- Crosswords & Games: 152 articles (0.05%)
- Education: 353 articles (0.12%)
- Fashion & Style: 279 articles (0.09%)
- Food: 133 articles (0.04%)
- Great Homes & Destinations: 3 articles (0.00%)
- Headway: 1 articles (0.00%)
- Home & Garden: 1 articles (0.00%)
- International Home: 1 articles (0.00%)
- Lens: 11 articles (0.00%)
- Magazine: 4 articles (0.00%)
- Movies: 677 articles (0.23%)
- Multimedia/Photos: 61 articles (0.02%)
- Obituaries: 1895 articles (0.63%)
- Opinion: 4 articles (0.00%)
- Parenting: 5 articles (0.00%)
- Special Series: 1 articles (0.00%)
- Sports: 31692 articles (10.58%)
- Sty

In [8]:
# Apply section filtering (include only relevant sections AND exclude NaN)
df_filtered = df[
    (df['section'].isin(INCLUDE_SECTIONS)) & 
    (df['section'].notna())
].copy()

print(f"\n=== FILTERING SUMMARY ===")
print(f"Original articles: {len(df)}")
print(f"After section filtering: {len(df_filtered)}")
print(f"Removed: {len(df) - len(df_filtered)} ({(len(df) - len(df_filtered))/len(df)*100:.2f}%)")

# Show remaining section distribution
print("\n=== REMAINING SECTIONS ===")
print(df_filtered['section'].value_counts())


=== FILTERING SUMMARY ===
Original articles: 299609
After section filtering: 209136
Removed: 90473 (30.20%)

=== REMAINING SECTIONS ===
section
New York          69610
Business Day      56914
Environment       55215
Technology        11990
U.S.               4376
World              2852
Your Money         2775
Real Estate        1422
Job Market         1098
The Upshot          838
Health              816
Automobiles         371
Science             363
Climate             253
Smarter Living      203
Washington           40
Name: count, dtype: int64


In [9]:
# Filter data from published_date >= 2008-01-01
df_filtered['published_date'] = pd.to_datetime(df_filtered['published_date'], errors='coerce')
df_filtered = df_filtered[df_filtered['published_date'] >= '2008-01-01']

In [10]:
# Save the included articles, create article_id from the index and save to a new CSV file
df_filtered['article_id'] = df_filtered.index
df_filtered.to_csv('data/01_preprocessed/articles_cleaned_sections_from_2008.csv', index=False)

In [11]:
df_filtered

,title,summary,section,keywords,published_date,url,article_id
99430,"Bloomberg in ’08? If So, Paper Chase Starts Soon",Money could smooth the way toward getting on t...,New York,"['Bloomberg, Michael R', 'Presidential Electio...",2008-01-01 05:00:00+00:00,https://www.nytimes.com/2008/01/01/nyregion/01...,99430
99431,"Boy’s Brother, 16, Believes Bullet Was Meant f...",The 11-year-old Queens boy who was shot in the...,New York,"['Crime and Criminals', 'Queens (NYC)']",2008-01-01 05:00:00+00:00,https://www.nytimes.com/2008/01/01/nyregion/01...,99431
99432,Corzine to Tour State for Plan to Cut Debt Wit...,Gov. Jon S. Corzine is going on a campaign-sty...,New York,"['Tolls', 'Roads and Traffic', 'Corzine, Jon S...",2008-01-01 05:00:00+00:00,https://www.nytimes.com/2008/01/01/nyregion/01...,99432
99433,"Gunshot Wounds Girl, 3",A 3-year-old girl was grazed by a bullet as sh...,New York,"['Crime and Criminals', 'Bronx (NYC)']",2008-01-01 05:00:00+00:00,https://www.nytimes.com/2008/01/01/nyregion/01...,99433
99434,"Edward Brennan, Who Led Sears at Its Peak, Die...",Mr. Brennan became chief executive at Sears in...,Business Day,"['Brennan, Edward', 'RETAIL STORES AND TRADE']",2008-01-01 05:00:00+00:00,https://www.nytimes.com/2008/01/01/business/01...,99434
...,...,...,...,...,...,...,...
299604,Environmental groups demand EPA to start monit...,A new legal petition filed by more than 170 to...,Environment,"['US Environmental Protection Agency', 'Plasti...",2024-12-01 13:00:07+00:00,https://www.theguardian.com/environment/2024/d...,299604
299605,"Land degradation expanding by 1m sq km a year,...",Land degradation is expanding worldwide at the...,Environment,"['Desertification', 'Farming', 'Deforestation'...",2024-12-01 12:00:07+00:00,https://www.theguardian.com/environment/2024/d...,299605
299606,"‘If I’m sent to Japan, I’m not coming home’: j...",The humpback whales watched by Paul Watson fro...,Environment,"['Sea Shepherd Conservation Society', 'Whales'...",2024-12-01 08:00:01+00:00,https://www.theguardian.com/environment/2024/d...,299606
299607,Cheaper loans on table to urge UK motorists to...,There is “no route to net zero” that ignores t...,Environment,"['Electric, hybrid and low-emission cars', 'Au...",2024-12-01 07:00:50+00:00,https://www.theguardian.com/environment/2024/d...,299607


## Text repetitions

In [12]:
# Check the specific problematic articles
problematic_ids = [177755, 207059, 172718, 171579, 244118]

print("=== ANALYZING SPECIFIC PROBLEMATIC ARTICLES ===\n")

for article_id in problematic_ids:
    if article_id in df_filtered['article_id'].values:
        article = df_filtered[df_filtered['article_id'] == article_id].iloc[0]
        
        print(f"\n{'='*80}")
        print(f"ARTICLE ID: {article_id}")
        print(f"{'='*80}")
        print(f"Title: {article['title']}")
        print(f"\nSummary: {article['summary']}")
        print(f"\n--- Metadata ---")
        print(f"Section: {article['section']}")
        print(f"Published: {article['published_date']}")
        print(f"Title length: {len(str(article['title']))} chars")
        print(f"Summary length: {len(str(article['summary']))} chars")
        
        # Show combined text to see full repetition
        combined = str(article['title']) + ' ' + str(article['summary'])
        print(f"\n--- Combined Text (first 500 chars) ---")
        print(combined[:500])
        print("...")
        print(f"\n--- Combined Text (last 500 chars) ---")
        print(combined[-500:])
        
    else:
        print(f"\nArticle ID {article_id} not found in filtered dataset")

=== ANALYZING SPECIFIC PROBLEMATIC ARTICLES ===


ARTICLE ID: 177755
Title: Earnings From Walmart and More Big Retailers; Wall Street’s Annual Meetings

Summary: Walmart, Home Depot and Target are among the retailers posting results this week; Japan will report its G.D.P. on Wednesday; and the big Wall Street banks will hold their annual meetings with shareholders.

--- Metadata ---
Section: Business Day
Published: 2015-05-17 23:52:15+00:00
Title length: 75 chars
Summary length: 205 chars

--- Combined Text (first 500 chars) ---
Earnings From Walmart and More Big Retailers; Wall Street’s Annual Meetings Walmart, Home Depot and Target are among the retailers posting results this week; Japan will report its G.D.P. on Wednesday; and the big Wall Street banks will hold their annual meetings with shareholders.
...

--- Combined Text (last 500 chars) ---
Earnings From Walmart and More Big Retailers; Wall Street’s Annual Meetings Walmart, Home Depot and Target are among the retailers posting 

Most severe case was 177755, check.

In [13]:
# Check if the repetition is in the actual data or just in display
print("=== CHECKING RAW SUMMARY FIELD FOR ARTICLE 177755 ===\n")

article = df_filtered[df_filtered['article_id'] == 177755].iloc[0]

print(f"Title: {article['title']}")
print(f"\nRaw Summary:")
print(article['summary'])
print(f"\nSummary contains title? {article['title'] in str(article['summary'])}")

# Check for internal repetitions in summary
summary_text = str(article['summary'])
sentences = [s.strip() for s in summary_text.split(';')]
print(f"\nSummary split by semicolon:")
for i, sent in enumerate(sentences, 1):
    print(f"{i}. {sent}")

# Check if any sentence repeats
from collections import Counter
sentence_counts = Counter(sentences)
repeated = [sent for sent, count in sentence_counts.items() if count > 1]
if repeated:
    print(f"\n⚠️ Repeated sentences found:")
    for sent in repeated:
        print(f"  - {sent}")
else:
    print("\n✅ No repeated sentences in summary")

=== CHECKING RAW SUMMARY FIELD FOR ARTICLE 177755 ===

Title: Earnings From Walmart and More Big Retailers; Wall Street’s Annual Meetings

Raw Summary:
Walmart, Home Depot and Target are among the retailers posting results this week; Japan will report its G.D.P. on Wednesday; and the big Wall Street banks will hold their annual meetings with shareholders.

Summary contains title? False

Summary split by semicolon:
1. Walmart, Home Depot and Target are among the retailers posting results this week
2. Japan will report its G.D.P. on Wednesday
3. and the big Wall Street banks will hold their annual meetings with shareholders.

✅ No repeated sentences in summary


In [14]:
# Find articles with repetitions in the summary itself
print("=== FINDING ACTUAL REPETITIONS IN SUMMARY FIELD ===\n")

def has_internal_repetition(summary):
    """Check if summary has internal repetition"""
    if pd.isna(summary):
        return False, 0, None
    
    summary = str(summary)
    
    # Method 1: Check if exact half repeats
    if len(summary) > 100:
        mid = len(summary) // 2
        first_half = summary[:mid].lower().strip()
        second_half = summary[mid:].lower().strip()
        if first_half == second_half:
            return True, 100, "exact_half_duplication"
    
    # Method 2: Check for repeated sentences
    sentences = [s.strip() for s in re.split(r'[.!?;]+', summary) if len(s.strip()) > 15]
    if len(sentences) > 1:
        sentence_counts = Counter(s.lower() for s in sentences)
        max_count = max(sentence_counts.values())
        if max_count > 2:
            return True, max_count * 20, "repeated_sentences"
    
    # Method 3: Check for repeated 5-word phrases
    words = summary.split()
    if len(words) >= 10:
        phrases = [' '.join(words[i:i+5]).lower() for i in range(len(words) - 4)]
        phrase_counts = Counter(phrases)
        max_repeats = max(phrase_counts.values()) if phrases else 0
        num_repeated = sum(1 for count in phrase_counts.values() if count > 1)
        
        if max_repeats > 3 or num_repeated > 5:
            return True, max_repeats * 10 + num_repeated * 5, "repeated_phrases"
    
    return False, 0, None

# Apply to all articles
results = df_filtered['summary'].apply(has_internal_repetition)
df_filtered['has_summary_repetition'] = results.apply(lambda x: x[0])
df_filtered['repetition_score'] = results.apply(lambda x: x[1])
df_filtered['repetition_type'] = results.apply(lambda x: x[2])

articles_with_rep = df_filtered['has_summary_repetition'].sum()
print(f"Articles with internal summary repetition: {articles_with_rep} ({articles_with_rep/len(df_filtered)*100:.2f}%)")

if articles_with_rep > 0:
    print("\n=== ARTICLES WITH REPETITION ===")
    repetitive = df_filtered[df_filtered['has_summary_repetition']].nlargest(10, 'repetition_score')
    
    for idx, row in repetitive.iterrows():
        print(f"\nArticle ID: {row['article_id']}")
        print(f"Title: {row['title']}")
        print(f"Type: {row['repetition_type']}")
        print(f"Score: {row['repetition_score']}")
        print(f"Summary (first 300 chars): {str(row['summary'])[:300]}...")
        print(f"Summary (last 300 chars): ...{str(row['summary'])[-300:]}")

=== FINDING ACTUAL REPETITIONS IN SUMMARY FIELD ===

Articles with internal summary repetition: 1925 (1.42%)

=== ARTICLES WITH REPETITION ===

Article ID: 265080
Title: Rio+20 'Future we Want' draft text - exclusive copy of the document
Type: repeated_phrases
Score: 4190
Summary (first 300 chars): Our Common Vision 1. We, the heads of State and Government and high level representatives, having met at Rio de Janeiro, Brazil, from 20-22 June 2012, with full participation of civil society, renew our commitment to sustainable development, and to ensure the promotion of economically, socially and ...
Summary (last 300 chars): ...he Secretary-General to compile these commitments and facilitate access to other registries that have compiled commitments, in an internet-based registry. The registry should make information about the commitments fully transparent and accessible to the public, and it should be periodically updated.

Article ID: 293355
Title: First draft of Cop27 text: what it says

In [15]:
# Analyze the types of repetitive articles more carefully
print("=== ANALYZING REPETITIVE ARTICLES ===\n")

# Look at sections of repetitive articles
print("Sections with most repetitions:")
repetitive_sections = df_filtered[df_filtered['has_summary_repetition']]['section'].value_counts()
print(repetitive_sections)

# Check if these are specific types of content
print("\n=== Examples by Type ===")

# Check for transcripts (like article 262489)
df_filtered['is_transcript'] = df_filtered['title'].str.contains('transcript|interview', case=False, na=False)

# Check for live blogs (like article 259825)
df_filtered['is_liveblog'] = df_filtered['title'].str.contains('live blog|live coverage', case=False, na=False) | \
                              df_filtered['summary'].str.contains('live blog|live coverage', case=False, na=False)

# Check for documents/drafts (like articles 265080, 293355, 293330)
df_filtered['is_document'] = df_filtered['title'].str.contains('draft|document|text|report', case=False, na=False)

# Check for letters/opinion collections (like article 274183)
df_filtered['is_letters'] = df_filtered['title'].str.contains('letter|letters to the editor', case=False, na=False)

print("\nBreakdown of repetitive articles:")
repetitive = df_filtered[df_filtered['has_summary_repetition']]
print(f"- Transcripts: {repetitive['is_transcript'].sum()}")
print(f"- Live blogs: {repetitive['is_liveblog'].sum()}")
print(f"- Documents/drafts: {repetitive['is_document'].sum()}")
print(f"- Letters: {repetitive['is_letters'].sum()}")
print(f"- Other: {len(repetitive) - (repetitive['is_transcript'].sum() + repetitive['is_liveblog'].sum() + repetitive['is_document'].sum() + repetitive['is_letters'].sum())}")

=== ANALYZING REPETITIVE ARTICLES ===

Sections with most repetitions:
section
Environment    1925
Name: count, dtype: int64

=== Examples by Type ===

Breakdown of repetitive articles:
- Transcripts: 3
- Live blogs: 12
- Documents/drafts: 66
- Letters: 6
- Other: 1838


In [16]:
# Review different severity levels
print("\n=== REPETITION SEVERITY ANALYSIS ===\n")

# Distribution by score
print("Repetition score distribution:")
print(df_filtered[df_filtered['has_summary_repetition']]['repetition_score'].describe())

# Show different severity buckets
severe = df_filtered[df_filtered['repetition_score'] > 1000]
moderate = df_filtered[(df_filtered['repetition_score'] > 500) & (df_filtered['repetition_score'] <= 1000)]
mild = df_filtered[(df_filtered['repetition_score'] > 0) & (df_filtered['repetition_score'] <= 500)]

print(f"\nSevere (score > 1000): {len(severe)} articles ({len(severe)/len(df_filtered)*100:.2f}%)")
print(f"Moderate (500 < score <= 1000): {len(moderate)} articles ({len(moderate)/len(df_filtered)*100:.2f}%)")
print(f"Mild (0 < score <= 500): {len(mild)} articles ({len(mild)/len(df_filtered)*100:.2f}%)")

# Sample from each category
print("\n=== SEVERE EXAMPLES ===")
for idx, row in severe.iterrows():
    print(f"\nArticle ID: {row['article_id']}")
    print(f"Title: {row['title']}")
    print(f"Score: {row['repetition_score']}")
    print(f"Type: Transcript={row['is_transcript']}, LiveBlog={row['is_liveblog']}, Document={row['is_document']}")

print("\n=== MODERATE EXAMPLES ===")
for idx, row in moderate.head(3).iterrows():
    print(f"\nArticle ID: {row['article_id']}")
    print(f"Title: {row['title']}")
    print(f"Score: {row['repetition_score']}")
    print(f"Type: Transcript={row['is_transcript']}, LiveBlog={row['is_liveblog']}, Document={row['is_document']}")


=== REPETITION SEVERITY ANALYSIS ===

Repetition score distribution:
count    1925.000000
mean       90.389610
std       132.112332
min        45.000000
25%        55.000000
50%        65.000000
75%        90.000000
max      4190.000000
Name: repetition_score, dtype: float64

Severe (score > 1000): 5 articles (0.00%)
Moderate (500 < score <= 1000): 12 articles (0.01%)
Mild (0 < score <= 500): 1908 articles (1.40%)

=== SEVERE EXAMPLES ===

Article ID: 259825
Title: 'Climategate' report - live blog
Score: 1365
Type: Transcript=False, LiveBlog=True, Document=True

Article ID: 265080
Title: Rio+20 'Future we Want' draft text - exclusive copy of the document
Score: 4190
Type: Transcript=False, LiveBlog=False, Document=True

Article ID: 273614
Title: Exxon knew of climate change in 1981, email says – but it funded deniers for 27 more years
Score: 1210
Type: Transcript=False, LiveBlog=False, Document=False

Article ID: 274183
Title: Government’s energy policies are just blowing in the wind


Seem like repetitve data is not an issue in raw data, check later preprocessing step (especially when we merge title and summary). => no action here needed.

In [17]:
# Clean up repetition analysis columns first
temp_cols = ['has_summary_repetition', 'repetition_score', 'repetition_type', 
             'is_transcript', 'is_liveblog', 'is_document', 'is_letters']
df_filtered = df_filtered.drop(columns=[c for c in temp_cols if c in df_filtered.columns], errors='ignore')

## Other issues with long articles

In [18]:
# Check the problematic articles from previous test set
problematic_long_ids = [253898, 296646, 253767]

print("=== ANALYZING PROBLEMATIC LONG ARTICLES ===\n")

for article_id in problematic_long_ids:
    if article_id in df_filtered['article_id'].values:
        article = df_filtered[df_filtered['article_id'] == article_id].iloc[0]
        
        print(f"\n{'='*80}")
        print(f"ARTICLE ID: {article_id}")
        print(f"{'='*80}")
        print(f"Title: {article['title']}")
        print(f"Section: {article['section']}")
        print(f"Published: {article['published_date']}")
        print(f"\nTitle length: {len(str(article['title']))} chars")
        print(f"Summary length: {len(str(article['summary']))} chars")
        print(f"Total length: {len(str(article['title'])) + len(str(article['summary']))} chars")
        
        print(f"\n--- Title ---")
        print(article['title'])
        
        print(f"\n--- Summary (first 500 chars) ---")
        print(str(article['summary'])[:500])
        print("...")
        
        print(f"\n--- Summary (last 500 chars) ---")
        print("...")
        print(str(article['summary'])[-500:])
        
    else:
        print(f"\nArticle ID {article_id} not found in filtered dataset")

=== ANALYZING PROBLEMATIC LONG ARTICLES ===


ARTICLE ID: 253898
Title: Greenwatch: China closes Everest for clean-up
Section: Environment
Published: 2008-07-01 08:57:27+00:00

Title length: 45 chars
Summary length: 1738 chars
Total length: 1783 chars

--- Title ---
Greenwatch: China closes Everest for clean-up

--- Summary (first 500 chars) ---
China closes Everest for clean-up >>Source: The Independent China is planning to restrict access by climbers to the summit of Mount Everest - known in China as Mount Qomolangma - to allow environmental teams to carry out a huge clean-up of the world's highest rubbish dump. Can a cow hormone help save the environment? >>Source: New Scientist Does a much-maligned product from Monsanto, have hidden environmental benefits? An analysis of the bovine hormone somatotropin, which is given to cows to boo
...

--- Summary (last 500 chars) ---
...
ist White House officials refused to open e-mail from the US environmental protection agency that said global

In [19]:
# Analyze article length distribution
print("\n=== ANALYZING ARTICLE LENGTH DISTRIBUTION ===\n")

df_filtered['title_length'] = df_filtered['title'].fillna('').astype(str).str.len()
df_filtered['summary_length'] = df_filtered['summary'].fillna('').astype(str).str.len()
df_filtered['total_length'] = df_filtered['title_length'] + df_filtered['summary_length']

print("Length statistics:")
print(df_filtered[['title_length', 'summary_length', 'total_length']].describe())

print("\nPercentiles:")
for p in [50, 75, 90, 95, 99, 99.5]:
    val = df_filtered['total_length'].quantile(p/100)
    print(f"  {p}th percentile: {val:.0f} characters")

# Check where our problematic articles fall
print("\n=== PROBLEMATIC ARTICLES IN DISTRIBUTION ===")
for article_id in problematic_long_ids:
    if article_id in df_filtered['article_id'].values:
        length = df_filtered[df_filtered['article_id'] == article_id]['total_length'].values[0]
        percentile = (df_filtered['total_length'] < length).sum() / len(df_filtered) * 100
        print(f"Article {article_id}: {length} chars (top {100-percentile:.1f}%)")


=== ANALYZING ARTICLE LENGTH DISTRIBUTION ===

Length statistics:
        title_length  summary_length   total_length
count  135962.000000   135962.000000  135962.000000
mean       54.728233     1574.254468    1628.982701
std        16.095573     2632.539196    2637.594114
min         3.000000        1.000000      18.000000
25%        44.000000      134.000000     185.000000
50%        55.000000      161.000000     217.000000
75%        66.000000     2603.000000    2661.000000
max       213.000000   169092.000000  169159.000000

Percentiles:
  50th percentile: 217 characters
  75th percentile: 2661 characters
  90th percentile: 4970 characters
  95th percentile: 6307 characters
  99th percentile: 10745 characters
  99.5th percentile: 13394 characters

=== PROBLEMATIC ARTICLES IN DISTRIBUTION ===
Article 253898: 1783 chars (top 32.0%)
Article 296646: 8846 chars (top 1.8%)
Article 253767: 1555 chars (top 33.0%)


In [20]:
# Show examples at different length ranges
print("\n=== EXAMPLES AT DIFFERENT LENGTH RANGES ===\n")

length_ranges = [
    (0, 500, "Very short"),
    (500, 1000, "Short (typical)"),
    (1000, 1500, "Medium"),
    (1500, 2500, "Long"),
    (2500, 4000, "Very long"),
    (4000, 100000, "Extremely long")
]

for min_len, max_len, label in length_ranges:
    articles_in_range = df_filtered[
        (df_filtered['total_length'] >= min_len) & 
        (df_filtered['total_length'] < max_len)
    ]
    
    print(f"\n{'='*80}")
    print(f"{label} ({min_len}-{max_len} chars): {len(articles_in_range)} articles ({len(articles_in_range)/len(df_filtered)*100:.2f}%)")
    print('='*80)
    
    if len(articles_in_range) > 0:
        # Sample 3 random articles
        sample = articles_in_range.sample(min(3, len(articles_in_range)))
        for idx, row in sample.iterrows():
            print(f"\nArticle ID: {row['article_id']} | Length: {row['total_length']}")
            print(f"Title: {row['title'][:90]}...")
            print(f"Summary preview: {str(row['summary'])[:150]}...")


=== EXAMPLES AT DIFFERENT LENGTH RANGES ===


Very short (0-500 chars): 88369 articles (65.00%)

Article ID: 109922 | Length: 159
Title: Not Everyone Is Cheering as Wi-Fi Takes to the Air...
Summary preview: Wireless Internet service on airlines may become a new source of tension between passengers on packed planes....

Article ID: 167003 | Length: 182
Title: Hotels Embrace the Campus Nearby...
Summary preview: Large chains are increasingly joining established independent hotels by tailoring their decorations and amenities to attract college-related business....

Article ID: 142029 | Length: 255
Title: Fourth-Quarter Profit and Revenue Declined at The New York Times Company...
Summary preview: Fourth-quarter profit declined as rising subscription and digital advertising revenue at the company’s largest newspapers could not offset the continu...

Short (typical) (500-1000 chars): 783 articles (0.58%)

Article ID: 283288 | Length: 993
Title: Plastic technology for natural recycling...
S

Since there are many long articles, we continue with linking entities to naturally filter out irrelevant articles, and then consider the long articles later.

# Final Data Check

In [21]:
df_date = df.copy()
df_date['published_date'] = pd.to_datetime(
    df_date['published_date'], errors='coerce'
)

df_date = df_date[
    df_date['published_date'] >= '2008-01-01'
]

print("After date restriction:", len(df_date))

df_date_section = df_date[
    df_date['section'].isin(INCLUDE_SECTIONS) &
    df_date['section'].notna()
]

print("After date + section restriction:", len(df_date_section))

After date restriction: 192602
After date + section restriction: 135962
